In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import os
import glob
import shutil
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.patches import Circle
from astropy.table import Table

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
import astropy.units as u
from matplotlib.patches import Circle
from astropy.table import Table




def interactive_aperture_editor(table_file, i, zoom_factor=5,
output_path = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv',
clean = False):
    table = Table.read(table_file)
    row = table[i]
    galaxy_name = row['galaxy']
    if galaxy_name == "M51":
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_NGC5194_ACS_WFC_drc.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1433':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1512':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1672':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    image_files = [hst_file, f187_file, f300_file, cont_sub_file]
    ra, dec = row['ra'], row['dec']
    radius_arcsec = row['radius']*u.arcsec

    coord = SkyCoord(ra, dec, unit='deg')

    fig, axes = plt.subplots(2, 2, figsize=(6.5, 6.5))
    axes = axes.flatten()

    circles = []
    cutouts = []  # <-- added

    for ax, file in zip(axes, image_files):
        try:
            hdu = fits.open(file)['SCI']
        except:
            hdu = fits.open(file)[0]
        data = hdu.data
        wcs = WCS(hdu.header)

        x, y = wcs.world_to_pixel(coord)
        pixscale = np.mean(np.abs(wcs.pixel_scale_matrix.diagonal())) * 3600
        size_pix = int((zoom_factor * radius_arcsec.value) / pixscale)

        cutout = Cutout2D(data, (x, y), (size_pix, size_pix), wcs=wcs)
        cutouts.append(cutout)  # <-- added

        cut_data = cutout.data

        cy, cx = np.array(cut_data.shape) / 2

        ax.imshow(
            cut_data,
            origin='lower',
            cmap='brg',
            vmin=np.nanpercentile(cut_data, 5),
            vmax=np.nanpercentile(cut_data, 99)
        )

        ax.set_title(file.split('/')[-1], fontsize=8)

        r_pix = radius_arcsec.value / pixscale

        circ = Circle((cx, cy), r_pix, edgecolor='red', facecolor='none', lw=2)
        ax.add_patch(circ)

        circles.append((circ, pixscale))

    dragging = {'active': False, 'mode': None}

    def on_press(event):
        if event.inaxes is None:
            return
        
        ax = event.inaxes
        idx = list(axes).index(ax)

        circ, pixscale = circles[idx]

        x0, y0 = circ.center
        dx = event.xdata - x0
        dy = event.ydata - y0
        dist = np.sqrt(dx**2 + dy**2)

        if dist < circ.radius/2:
            dragging['mode'] = 'move'
        else:
            dragging['mode'] = 'resize'

        dragging['active'] = True

    def on_release(event):
        dragging['active'] = False
        dragging['mode'] = None

    def on_motion(event):
        if not dragging['active'] or event.inaxes is None:
            return

        ax = event.inaxes
        idx = list(axes).index(ax)

        circ, pixscale = circles[idx]

        x0, y0 = circ.center

        if dragging['mode'] == 'resize':
            dx = event.xdata - x0
            dy = event.ydata - y0
            new_r = np.sqrt(dx**2 + dy**2)

            for c, _ in circles:
                c.set_radius(new_r)

        elif dragging['mode'] == 'move':
            new_center = (event.xdata, event.ydata)

            for c, _ in circles:
                c.set_center(new_center)

        fig.canvas.draw_idle()

    def on_key(event):
        if event.key == 'enter':
            r_pix = circles[0][0].get_radius()
            pixscale = circles[0][1]
            new_radius = r_pix * pixscale

            # --- NEW: update RA/Dec ---
            x_pix, y_pix = circles[0][0].center
            sky = cutouts[0].wcs.pixel_to_world(x_pix, y_pix)
            table[i]['ra'] = sky.ra.deg
            table[i]['dec'] = sky.dec.deg
            # -------------------------

            table[i]['radius'] = new_radius
            try:
                table.write(output_path, format='csv', overwrite=True)
                print(f"Table successfully saved to {output_path}")
            except Exception as e:
                print(f"Error saving table: {e}")
            print(f"Updated row {i} radius → {new_radius:.3f} arcsec")
            plt.close(fig)

        elif event.key == 'b':
            table[i]['is_bad'] = 1
            r_pix = circles[0][0].get_radius()
            pixscale = circles[0][1]
            new_radius = r_pix * pixscale

            # --- NEW ---
            x_pix, y_pix = circles[0][0].center
            sky = cutouts[0].wcs.pixel_to_world(x_pix, y_pix)
            table[i]['ra'] = sky.ra.deg
            table[i]['dec'] = sky.dec.deg
            # ------------

            table[i]['radius'] = new_radius
            try:
                table.write(output_path, format='csv', overwrite=True)
                print(f"Table successfully saved to {output_path}")
            except Exception as e:
                print(f"Error saving table: {e}")
            print(f"flagged row {i} as BAD")

        elif event.key == 'm':
            table[i]['is_bad'] = 2
            r_pix = circles[0][0].get_radius()
            pixscale = circles[0][1]
            new_radius = r_pix * pixscale

            # --- NEW ---
            x_pix, y_pix = circles[0][0].center
            sky = cutouts[0].wcs.pixel_to_world(x_pix, y_pix)
            table[i]['ra'] = sky.ra.deg
            table[i]['dec'] = sky.dec.deg
            # ------------

            table[i]['radius'] = new_radius
            try:
                table.write(output_path, format='csv', overwrite=True)
                print(f"Table successfully saved to {output_path}")
            except Exception as e:
                print(f"Error saving table: {e}")
            print(f"flagged row {i} as MAYBE")

    fig.canvas.mpl_connect('button_press_event', on_press)
    fig.canvas.mpl_connect('button_release_event', on_release)
    fig.canvas.mpl_connect('motion_notify_event', on_motion)
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()

    log_files = glob.glob('/project/galaxies/tjuchau/data_files/misc_data/progress_log/*')
    if clean:
        for file in log_files:
            os.rmdir(file)
        os.mkdir(f'/project/galaxies/tjuchau/data_files/misc_data/progress_log/Last_row{i}')
import warnings
from astropy.wcs import FITSFixedWarning

warnings.simplefilter('ignore', FITSFixedWarning)
glob.glob('/project/galaxies/tjuchau/data_files/misc_data/progress_log/*')

In [ ]:
table_file = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv'
interactive_aperture_editor(table_file, i, clean=True)
i+=1

In [ ]:
temp = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
temp[i-3:i+1]

In [ ]:
temp = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
temp[i]

In [ ]:
i-=1

In [ ]:
i

In [ ]:
from Functions import *
temp = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
for i, row in enumerate(temp):
    if i > int(glob.glob('/project/galaxies/tjuchau/data_files/misc_data/progress_log/*')[0][-2:]):
        break
    
    if row['is_bad'] == 0:
        print(f'Below image {i} was marked as good')
    if row['is_bad'] == 1:
        print(f'Below image {i} was marked as BAD')
    if row['is_bad'] == 2:
        print(f'Below image {i} was marked as MAYBE OK')
    galaxy_name = row['galaxy']
    if galaxy_name == "M51":
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_NGC5194_ACS_WFC_drc.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1433':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1512':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    elif galaxy_name == 'ngc1672':
        hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'
        f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits'
        f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f300m_i2d_anchor.fits'
        cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    image_files = [hst_file, f187_file, f300_file, cont_sub_file]
    ra, dec = row['ra'], row['dec']
    radius_arcsec = row['radius']*u.arcsec
    show_images(image_files, [ra,dec], radius_arcsec, ncols=4)

In [ ]:
i = 8
table_file = '/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv'
interactive_aperture_editor(table_file, i, clean=True)